In [1]:
import numpy as np
import matplotlib.pyplot as plt

def read_data_file(filename):
    data   = []
    values = []
    with open(filename, 'r') as file:
        for line in file:
            values.append(line.strip().split())
        for i in range(0, len(values), 2):
            # Extract the two consecutive lines
            line1 = values[i]
            line2 = values[i + 1]
            # Create a tuple of four elements and append to the result list
            if len(line1) == 3 and len(line2) == 1:
                array_data = line1 + line2
                data.append(array_data)
    return np.array(data)

def parse_config_file(filepath):
    config = {}
    with open(filepath, 'r') as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        label_line = lines[i].strip().strip("'").strip('"')
        i += 1
        if i >= len(lines):
            break
        value_line = lines[i].strip()
        i += 1
        labels = label_line.split()
        values = value_line.split()
        def try_convert(val):
            try:
                if '.' in val or 'e' in val.lower():
                    return float(val)
                else:
                    return int(val)
            except ValueError:
                return val
        converted_values = list(map(try_convert, values))
        if len(labels) == 1:
            config[labels[0]] = converted_values[0] if len(converted_values) == 1 else converted_values
        else:
            config.update(dict(zip(labels, converted_values)))
    return config

if __name__ == '__main__':
    filepath = 'gemx.in'
    parameters_dict = parse_config_file(filepath)

dt      = parameters_dict['dt']
omegaA  = 1539912.119669488
psi_p       = np.loadtxt('psi_p.dat')
ni_dat      = np.loadtxt('ni0_gorler.dat')
Ti_dat      = np.loadtxt('ti0_gorler.dat')  

phirphi     = np.loadtxt('testphi_r_phi')
phidiag     = np.loadtxt('testPhiDiag')
phidiag_fq  = np.loadtxt('testPhiFreq')
phidiag_fq2 = np.loadtxt('testPhiFreq2')
#ttt = np.loadtxt('testAtPhitrhotjt') 
# ttt         = read_data_file('testAtPhitrhotjt')
phirphi     = np.loadtxt('testphi_r_phi')  
ex          = np.loadtxt('testER')
ez          = np.loadtxt('testEZ')
R           = np.loadtxt('R.dat') 
Z           = np.loadtxt('Z.dat')  
dR          = (R[-1] - R[0]) / (len(R) - 1)
dZ          = (Z[-1] - Z[0]) / (len(Z) - 1)
BR          = np.loadtxt('BR.dat') 
BZ          = np.loadtxt('Bz.dat')  
Bt          = np.loadtxt('Bt.dat')
mask2       = np.loadtxt('mask2') 
mask3       = np.loadtxt('mask3')
mask4       = np.loadtxt('mask4')
testphi     = np.loadtxt('testphi')
testphiavg  = np.loadtxt('testphiavg')
masktest    = np.loadtxt('mask')   
netest      = np.loadtxt('testne0') 
apar0       = np.loadtxt('testapar0')
ne2         = np.loadtxt('testne')  
apar2       = np.loadtxt('testapar')
jpar2       = np.loadtxt('testjpar')
jpar0       = np.loadtxt('testj0')
ne0zeta     = np.loadtxt('testne0_zeta')
j0zeta      = np.loadtxt('testjpar0_zeta')
c2_over_vA2 = np.loadtxt('test_1_over_vA2')
weights     = np.loadtxt('testweights')
fluxes      = np.loadtxt('test_fluxdiag')

imx = parameters_dict['imx']+1
jmx = parameters_dict['jmx']+1
kmx = parameters_dict['kmx']+1

p = np.linspace(0, 2*np.pi, 258, endpoint=False)

# Initialize arrays to store loaded data (assuming the same dimensions as Matlab script)
phi2  = np.zeros((imx, jmx))
# tmp   = np.zeros((imx, jmx))
# out   = np.zeros((imx,jmx))
aphi  = np.zeros((imx, jmx))
ER    = np.zeros((imx,jmx))
EZ    = np.zeros((imx,jmx))
a0    = np.zeros((imx, jmx))
m1    = np.zeros((imx, jmx))
psi   = np.zeros((imx, jmx))
n0    = np.zeros((imx, jmx))
n2    = np.zeros((imx, jmx))
a2    = np.zeros((imx, jmx))
j0    = np.zeros((imx, jmx))
j2    = np.zeros((imx, jmx))
m2    = np.zeros((imx, jmx))
m3    = np.zeros((imx, jmx))
m4    = np.zeros((imx, jmx))
vA2   = np.zeros((imx, jmx))
prphi = np.zeros((kmx, imx))

# Loop to populate arrays with loaded data
for i in range(imx):
    for j in range(jmx):
        ii = ((i-1)*imx + j-1) // 3
        jj = (i-1)*jmx + j - 3 * (((i-1)*imx + j-1) // 3) - 1 

        phi2[i, j] = testphi[ii, jj]
        # out[i,j]   = testout[ii,jj]
        aphi[i, j] = testphiavg[ii, jj]
        ER[i, j]   = ex[ii, jj]
        EZ[i, j]   = ez[ii, jj]
        a0[i, j] = apar0[ii, jj]
        m1[i, j] = masktest[ii, jj]
        #psi[i, j] = psitest[ii, jj]
        n0[i, j] = netest[ii, jj]
        n2[i, j] = ne2[ii, jj]
        a2[i, j] = apar2[ii, jj]
        j0[i, j] = jpar0[ii, jj]
        j2[i, j] = jpar2[ii, jj]
        m2[i, j] = mask2[ii, jj]
        m3[i, j] = mask3[ii, jj]
        m4[i, j] = mask4[ii, jj]
        vA2[i, j] = c2_over_vA2[ii, jj]

# Loop to populate prphi array
for i in range(kmx):
    for j in range(imx):
        ii = ((i-1)*imx + j-1) // 3
        jj = (i-1)*imx + j - 3 * (((i-1)*imx + j-1) // 3) - 1
        prphi[i, j] = phirphi[ii, jj]

#######################################################
#SET SATURATION MARKER
sat_marker = 1500
end_marker = 5000
#######################################################

gamma       = (1/2)*np.gradient(np.log(phidiag[:,1]),dt)
# growth_rate = np.sum(gamma[sat_marker:])/len(gamma[sat_marker:])

# print(f"Saturated Growth Rate: {growth_rate*10**(-4)} 10^-4 s")

# Real Frequency Calculation
if phidiag[-1,0] >= sat_marker:

    growth_rate = np.sum(gamma[sat_marker:end_marker])/len(gamma[sat_marker:end_marker])
    fq_array    = np.array([phidiag_fq[sat_marker:end_marker,1],phidiag_fq[sat_marker:end_marker,2],phidiag_fq2[sat_marker:end_marker,1],phidiag_fq2[sat_marker:end_marker,2]])
    omega_array = []
    peak_omega  = []
    
    for i in range(len(fq_array)):
        tmp_freq    = fq_array[i]*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate)
        fft_phi     = np.fft.fft(tmp_freq)
        freq        = np.fft.fftfreq(len(tmp_freq), d=dt)
        freqindex   = np.arange(len(tmp_freq)//2)
        fft_phi_mag = np.abs(fft_phi[freqindex])
        freq_pos    = freq[freqindex]
        omega_array.append(freq_pos)
        peak_omega.append(freq_pos[np.argmax(fft_phi_mag)])

        print(f"Dominant Omega_r: {peak_omega[i]*10**(-4)*2*np.pi} /(10^4 rad/s)")

    print(f"Saturated Growth Rate: {growth_rate*10**(-4)} 10^-4 s")    
    freq_total = (phidiag_fq[sat_marker:end_marker,1]+phidiag_fq[sat_marker:end_marker,2]+phidiag_fq2[sat_marker:end_marker,1]+phidiag_fq2[sat_marker:end_marker,2])*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate)


# Diffusion Coefficent Calculation (Using Equillibrium Gradients, Maybe Wrong)
heat_flux     = fluxes[:,1]
particle_flux = fluxes[:,2]

kappa_n = 2.23
kappa_T = 6.96

diffusion_D   = particle_flux/(kappa_n*ni_dat[192,128])
diffusion_chi = heat_flux/(kappa_T*ni_dat[192,128]*Ti_dat[192,128])

plt.figure()
# plt.countourfx,y)

phi_mesh = np.linspace(0, 2*np.pi, kmx)

r, theta = np.meshgrid(R, phi_mesh)

#-- Plot... ------------------------------------------------
fig, ax = plt.subplots(subplot_kw=dict(projection='polar'))
ax.contourf(theta, r, prphi,100)
ax.set_ylim(0,np.max(R))
plt.savefig("polar.png")
plt.show()
        
# Plotting figures
plt.figure()
plt.plot(weights[:,0],weights[:,1])
plt.xlabel('timestep')
plt.ylabel('$\\frac{\\sum w_p}{N}$')
# plt.savefig("weight_diag.png")
plt.grid()
plt.show()

# freq_total = (phidiag_fq[sat_marker:end_marker,1]+phidiag_fq[sat_marker:end_marker,2]+phidiag_fq2[sat_marker:end_marker,1]+phidiag_fq2[sat_marker:end_marker,2])*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate)

# plt.figure()
# # plt.plot(phidiag_fq[sat_marker:end_marker,0],phidiag_fq[sat_marker:end_marker,1]*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate),label='R = ' + str(R[192]))
# # plt.plot(phidiag_fq[sat_marker:end_marker,0],phidiag_fq[sat_marker:end_marker,2]*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate),label='R = ' + str(R[180]))
# # plt.plot(phidiag_fq[sat_marker:end_marker,0],phidiag_fq2[sat_marker:end_marker,1]*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate),label='R = ' + str(R[170]))
# # plt.plot(phidiag_fq[sat_marker:end_marker,0],phidiag_fq2[sat_marker:end_marker,2]*np.exp(-dt*phidiag_fq[sat_marker:end_marker,0]*growth_rate),label='R = ' + str(R[200]))
# plt.plot(phidiag_fq[sat_marker:end_marker,0],freq_total,label='total')
# # plt.plot(phidiag_fq[:,0],phidiag_fq[:,1],label=str(R[192]))
# # plt.plot(phidiag_fq[:,0],phidiag_fq[:,2],label=str(R[180]))
# # plt.plot(phidiag_fq2[:,0],phidiag_fq2[:,1],label=str(R[170]))
# # plt.plot(phidiag_fq2[:,0],phidiag_fq2[:,2],label=str(R[200]))
# # plt.plot(phidiag_fq[:,0],phidiag_fq[:,2])
# plt.legend()
# plt.xlabel('timestep')
# plt.ylabel('$\\phi$')
# # plt.savefig("phisquare.png")
# plt.grid()
# plt.show()

if phidiag[-1,0] >= sat_marker:
    plt.figure()
    plt.plot(freq_pos*10**(-4)*2*np.pi, fft_phi_mag)
    plt.xlabel("$\\omega_r / 10^4 $rad/s")
    plt.ylabel("Amplitude")
    plt.title("FFT of $\\phi$")
    plt.xlim(0,60)
    plt.grid()
    plt.show()

plt.figure()
plt.plot(phidiag[:,0],phidiag[:,1])
plt.xlabel('timestep')
plt.ylabel('$|\\phi|^2$')
plt.savefig("phisquare.png")
plt.grid()
plt.show()

plt.figure()
plt.plot(phidiag[:,0],np.log(phidiag[:,1]))
plt.xlabel('timestep')
plt.ylabel('$ln(|\\phi|^2)$')
plt.savefig("phisquare.png")
plt.grid()
plt.show()

plt.figure()
plt.plot(phidiag[:,0],gamma)
plt.xlabel('timestep')
plt.ylabel('$\\gamma$')
plt.savefig("gamma.png")
plt.grid()
plt.show()

plt.figure()
plt.plot(fluxes[:,0],heat_flux/max(abs(heat_flux)),label='Q')
plt.plot(fluxes[:,0],particle_flux/max(abs(particle_flux)),label='$\\Gamma$')
# plt.plot(fluxes[:,0],fluxes[:,3]/max(fluxes[:,3]),label='$\\Pi$')
plt.legend()
plt.xlabel('timestep')
plt.ylabel('Norm. Flux')
plt.grid()
plt.savefig('fluxes.png')
plt.show()

plt.figure()
plt.contourf(R, Z, n2, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$n_{e}$', fontsize=24)
# plt.savefig('Figures/n_e.png')

# plt.figure()
# plt.contourf(R, Z, tmp, 100)
# #plt.axis('equal')
# plt.colorbar()
# plt.xlabel('R(m)', fontsize=20)
# plt.ylabel('Z(m)',fontsize=20)
# plt.title(r'$RHS$', fontsize=24)

plt.figure()
plt.contourf(R, Z, a2, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$\delta A_{||}$', fontsize=24)
# plt.savefig('Figures/delApar.png')

plt.figure()
plt.contourf(R, Z, j2, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$\mu_0 \delta j_{e||}$', fontsize=24)
# plt.savefig('Figures/mudelj.png')

plt.figure()
plt.contourf(R, Z, n0, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$n_{e0}$', fontsize=24)
# plt.savefig('Figures/n_e0.png')

plt.figure()
plt.contourf(R, Z, a0, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$A_{||0}$', fontsize=24)
# plt.savefig('Figures/Apar0.png')

plt.figure()
plt.contourf(R, Z, phi2, 100)#,cmap='RdYlBu')
plt.colorbar()
plt.contour(R, Z, m1 * 0.001, 1, colors='black', linewidths=2)
plt.scatter(R[192],Z[128])
plt.scatter(R[180],Z[128])
plt.scatter(R[170],Z[128])
plt.scatter(R[200],Z[128])
#plt.axis('equal')
#plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$\phi_{end}$', fontsize=24)
# plt.savefig('Figures/phiend.png')

# plt.figure()
# plt.contourf(R, Z, out, 100)
# plt.colorbar()
# plt.contour(R, Z, m1 * 0.001, 1, colors='black', linewidths=2)
# plt.axis('equal')
# #plt.colorbar()
# plt.xlabel('R(m)', fontsize=20)
# plt.ylabel('Z(m)',fontsize=20)
# plt.title(r'out', fontsize=24)

plt.figure()
plt.contourf(R, Z, aphi, 100)#,cmap='RdYlBu')
plt.colorbar()
plt.contour(R,Z,m1 * 0.001, 1, colors='black', linewidths=2)
#plt.axis('equal')
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)', fontsize=20)
plt.title(r'$\left < \phi_{end} \right >$', fontsize=24)
plt.savefig('avgphiend.png')

plt.figure()
plt.contourf(R, Z, j0, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$j_{e||0}$', fontsize=24)
# plt.savefig('Figures/jpar0.png')

plt.figure()
plt.contourf(R, Z, vA2, 100)#,cmap='RdYlBu')
#plt.axis('equal')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$1/v_A^2$', fontsize=24)
# plt.savefig('Figures/1_vA2.png')

plt.figure()
plt.contourf(R, Z, ER, 100)#,cmap='RdYlBu')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$E_R$', fontsize=24)
# plt.savefig('Figures/E_R.png')

plt.figure()
plt.contourf(R, Z, EZ, 100)#,cmap='RdYlBu')
plt.colorbar()
plt.xlabel('R(m)', fontsize=20)
plt.ylabel('Z(m)',fontsize=20)
plt.title(r'$E_Z$', fontsize=24)
# plt.savefig('Figures/E_Z.png')

plt.show()

In [2]:

import numpy as np
import matplotlib.pyplot as plt

testfunc = np.loadtxt('testfourier')
filtered = np.loadtxt('testfourier_filtered')

x = np.linspace(0,2*np.pi,500)
x2 = np.linspace(0,2*np.pi,500)
print(max(filtered)/max(testfunc))

plt.plot(x,testfunc/max(testfunc))
plt.plot(x2,filtered/max(filtered))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
# import imageio
# import cv2

# Constants
dt = 500e-10
steps = 200
omegaA = 1539912.119669488
filename = "testAnimated.gif"

# Load data
psi_p = np.loadtxt('psi_p.dat')  # 2D psi grid
phirphi = np.loadtxt('testphi_r_phi')  # Unused here
R = np.loadtxt('R.dat')
Z = np.loadtxt('Z.dat')
BR = np.loadtxt('BR.dat')
BZ = np.loadtxt('Bz.dat')
Bt = np.loadtxt('Bt.dat')
mask2 = np.loadtxt('mask2')
mask3 = np.loadtxt('mask3')
mask4 = np.loadtxt('mask4')
phit = np.loadtxt('testPhit')  # Large 2D array
mask = np.loadtxt('mask')      # Used to construct m1

# Construct m1 mask
m1 = np.zeros((258, 258))
for i in range(258):
    for j in range(258):
        ii = ((i * 258 + j) // 3) + 1
        jj = (i * 258 + j) - 3 * (((i * 258 + j) // 3))
        m1[i, j] = mask[ii-1, jj]

# Get psi contour data
fig = plt.figure()
psi_contours = plt.contour(R,Z,psi_p,40,levels=np.linspace(0,0.2,40))
plt.close(fig)

psi_values = psi_contours.levels

psi_paths = []
for collection in psi_contours.collections:
    for path in collection.get_paths():
        v = path.vertices  # shape (N, 2) --> [R, Z] pairs
        psi_paths.append(v)

# Create the GIF
images = []
for idx in range(steps):
    if (idx*100+100 > 1250):
        break
    pt = np.zeros((258, 258))
    for i in range(258):
        for j in range(258):
            step_data = phit[idx * 22188 : (idx + 1) * 22188, :]
            ii = ((i * 258 + j) // 3) + 1
            jj = (i * 258 + j) - 3 * (((i * 258 + j) // 3))
            if (idx + 1) * 22188 > phit.shape[0]:
                break
            pt[i, j] = phit[ii-1 + (idx * 22188), jj]
            idx_phit = ii-1 + (idx * 22188)
    
    # Create interpolator for this frame
    # pt_interp = RegularGridInterpolator((R, Z), pt.T)  # Note the transpose
    pt_interp = RegularGridInterpolator((R,Z), pt)

    # Sample pt along each psi_p contour path
    pt_along_psi = []
    for path in psi_paths:
        try:
            values = pt_interp(path)  # path = Nx2 array of [R, Z]
        except ValueError:
            # Some points might be out of bounds
            values = np.full(path.shape[0], np.nan)
        pt_along_psi.append(values)
    
    phi_contour_diag = []
    for contour in range(len(psi_paths)):
        # print(contour)
        phi_contour_diag.append(np.sum(pt_along_psi[contour]**2)/len(pt_along_psi[contour]))
        

    # --- Plot ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Main 2D plot
    c1 = ax1.contourf(R, Z, pt, 20, cmap='viridis')
    ax1.contour(R, Z, m1 * 0.001, 1, colors='black', linewidths=2)
    ax1.set_title(f'Time Step {idx*100+100}')
    ax1.set_aspect('equal')
    fig.colorbar(c1, ax=ax1)

    # Average phi vs normalized psi
    ax2.plot(psi_values[1:]/np.max(psi_values), phi_contour_diag, color='blue')
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('$\\psi_{Norm}$')
    ax2.set_ylabel('$|\\phi|^2$')
    # ax2.set_title('Flux-Surface Averaged ϕ')
    ax2.grid(True)

    # Save frame to image buffer
    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    images.append(frame)
    # plt.close(fig)

# Save GIF
imageio.mimsave(filename, images, format='GIF', duration=0.3)
